# TensorBoard and Run Analysis

Post-run analysis of a training directory. The trainer writes:

- `<run>/tb_events/` - a single TensorFlow event log with all scalars and image
  summaries, tagged by section (`train/`, `val/`, `epoch/`, `system/`, and
  per-class `cls/<NN>_<name>/<metric>`).
- `<run>/val_history.jsonl` - one JSON line per validation, the full per-category
  report plus headline metrics (`mAP`, `mAP50`, `F1score50`, `AR100`).

This notebook loads both, plots the training and validation curves, and reads the
per-class best-F1 trends and best-epoch selection from the JSONL store. Set
`RUN_DIR` below.

## Parameters

In [ ]:
# Output directory of a training run (contains tb_events/ and val_history.jsonl).
RUN_DIR = "/tmp/yolo_run"

import os
assert os.path.isdir("eval"), "Run from the repo root (directory containing eval/, train/, ...)."
print("run dir:", RUN_DIR)
print("  tb_events         :", os.path.isdir(os.path.join(RUN_DIR, "tb_events")))
print("  val_history.jsonl :", os.path.isfile(os.path.join(RUN_DIR, "val_history.jsonl")))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## Load the event log

`EventAccumulator` (shipped with TensorFlow's bundled TensorBoard) reads the
event files without launching the TensorBoard server. Loading the full log lazily
can be slow on long runs; the size guidance below keeps scalars uncapped.

In [ ]:
try:
    from tensorboard.backend.event_processing.event_accumulator import (
        EventAccumulator, SCALARS,
    )
    _HAVE_TB = True
except ImportError:
    _HAVE_TB = False
    print("tensorboard not importable - install it or use `pip install tensorboard`.")

tb_dir = os.path.join(RUN_DIR, "tb_events")
ea = None
if _HAVE_TB and os.path.isdir(tb_dir):
    # size_guidance 0 = keep everything for the requested type.
    ea = EventAccumulator(tb_dir, size_guidance={SCALARS: 0})
    ea.Reload()
    scalar_tags = sorted(ea.Tags().get("scalars", []))
    print(f"{len(scalar_tags)} scalar tags\n")
    for t in scalar_tags:
        print(" ", t)
else:
    scalar_tags = []
    print("No event log found at", tb_dir)

In [ ]:
def scalar_series(tag):
    '''Return (steps, values) arrays for a scalar tag, or empty arrays.'''
    if ea is None or tag not in scalar_tags:
        return np.array([]), np.array([])
    events = ea.Scalars(tag)
    return (np.array([e.step for e in events]),
            np.array([e.value for e in events]))

## Training losses

The headline loss components under `train/` (gain-weighted unless noted). See `eval/metric_meta.py` for the exact formula behind each tag.

In [ ]:
from eval.metric_meta import describe

loss_tags = ["train/total_loss", "train/box_loss", "train/dfl_loss",
             "train/cls_loss", "train/dist_loss", "train/poly_loss",
             "train/poly_angle_loss", "train/poly_dist_loss", "train/poly_conf_loss"]
present = [t for t in loss_tags if t in scalar_tags]

if present:
    cols = 3
    rows = (len(present) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(15, 4 * rows))
    for ax, tag in zip(np.array(axes).reshape(-1), present):
        s, v = scalar_series(tag)
        ax.plot(s, v, linewidth=1)
        ax.set_title(tag.split("/", 1)[1]); ax.set_xlabel("step"); ax.grid(alpha=0.3)
    for ax in np.array(axes).reshape(-1)[len(present):]:
        ax.axis("off")
    plt.tight_layout(); plt.show()

    # Short definition of the primary loss (markdown from the metric registry).
    d = describe("total_loss")
    if d:
        print("total_loss:", d)
else:
    print("No train/*_loss tags found.")

## Learning rate and gradient norm

`train/lr` (cosine decay after warmup) and `train/grad_norm` (global L2 norm before clipping - watch for spikes).

In [ ]:
diag = [("train/lr", "learning rate"), ("train/grad_norm", "grad norm (pre-clip)"),
        ("train/weight_norm", "weight norm"), ("train/update_ratio", "update/weight ratio")]
diag = [(t, lbl) for t, lbl in diag if t in scalar_tags]

if diag:
    fig, axes = plt.subplots(1, len(diag), figsize=(6 * len(diag), 4))
    for ax, (tag, lbl) in zip(np.atleast_1d(axes), diag):
        s, v = scalar_series(tag)
        ax.plot(s, v, linewidth=1)
        ax.set_title(lbl); ax.set_xlabel("step"); ax.grid(alpha=0.3)
    plt.tight_layout(); plt.show()
else:
    print("No lr / grad_norm tags found.")

## Validation metric curves

Headline detection, polygon, and distance metrics under `val/`, plotted over training steps.

In [ ]:
val_tags = ["val/mAP", "val/mAP50", "val/F1score50", "val/AR100",
            "val/poly_mIoU", "val/poly_recall50", "val/dist_mae", "val/dist_absrel"]
present = [t for t in val_tags if t in scalar_tags]

if present:
    cols = 4
    rows = (len(present) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(16, 4 * rows))
    for ax, tag in zip(np.array(axes).reshape(-1), present):
        s, v = scalar_series(tag)
        ax.plot(s, v, marker="o", markersize=3, linewidth=1)
        ax.set_title(tag.split("/", 1)[1]); ax.set_xlabel("step"); ax.grid(alpha=0.3)
    for ax in np.array(axes).reshape(-1)[len(present):]:
        ax.axis("off")
    plt.tight_layout(); plt.show()
else:
    print("No val/* tags found (has validation run yet?).")

## val_history.jsonl - best epoch selection

The JSONL store is the authoritative per-epoch record. `eval/val_history.py`
collapses re-validations to one row per epoch, extracts the headline metric, and
selects the best epoch by `F1score50`.

In [ ]:
from eval.val_history import (
    resolve_path, load_records, latest_per_epoch, best_record, metric_of,
)

jsonl_path = resolve_path(RUN_DIR)
records = latest_per_epoch(load_records(jsonl_path))
print(f"{len(records)} epoch record(s) in {jsonl_path}")

if records:
    print(f"\n{'epoch':>6} {'step':>9} {'F1@50':>8} {'mAP':>8} {'mAP50':>8} {'AR100':>8}")
    for r in records:
        m = r.get("metrics", {})
        print(f"{r.get('epoch', -1):>6} {r.get('step', -1):>9} "
              f"{metric_of(r, 'F1score50') or float('nan'):>8.4f} "
              f"{m.get('mAP', float('nan')):>8.4f} {m.get('mAP50', float('nan')):>8.4f} "
              f"{m.get('AR100', float('nan')):>8.4f}")

    best = best_record(records, key="F1score50")
    if best is not None:
        print(f"\nBest epoch by F1score50: epoch={best.get('epoch')} "
              f"step={best.get('step')} F1={metric_of(best, 'F1score50'):.4f} "
              f"checkpoint={best.get('checkpoint')}")

## Per-class best-F1 trends

Each record's `best_conf` table lists, per category, the peak F1 (and the
confidence at which it peaks). We pivot it into a class-by-epoch matrix and plot
the trend for the lowest-scoring tail classes - the ones that move the macro
`F1score50` most.

In [ ]:
from configs.class_map import DETECTION_CLASSES

if records:
    epochs = [r.get("epoch", i) for i, r in enumerate(records)]
    # category -> list of best-F1 per epoch (NaN where absent).
    per_class = {}
    for r in records:
        seen = {int(row["category"]): float(row["f1"]) for row in r.get("best_conf", [])}
        for cat in seen:
            per_class.setdefault(cat, [np.nan] * len(records))
    for e_idx, r in enumerate(records):
        for row in r.get("best_conf", []):
            per_class.setdefault(int(row["category"]), [np.nan] * len(records))
            per_class[int(row["category"])][e_idx] = float(row["f1"])

    # Rank classes by their last-epoch F1; plot the weakest few.
    def _last(vals):
        v = [x for x in vals if not np.isnan(x)]
        return v[-1] if v else np.nan
    ranked = sorted(per_class.items(), key=lambda kv: (_last(kv[1]) if not np.isnan(_last(kv[1])) else 1e9))
    worst = ranked[:6]

    fig, ax = plt.subplots(figsize=(11, 6))
    for cat, vals in worst:
        name = DETECTION_CLASSES.get(cat, f"class_{cat}")
        ax.plot(epochs, vals, marker="o", markersize=3, label=f"{cat:02d} {name}")
    ax.set_xlabel("epoch"); ax.set_ylabel("best F1@50"); ax.grid(alpha=0.3)
    ax.set_title("Per-class best-F1 trend (6 weakest classes at the latest epoch)")
    ax.legend(fontsize=8, ncol=2)
    plt.tight_layout(); plt.show()
else:
    print("No records to pivot.")